### GBIF Data Exploration: Great-tailed Grackle (*Quiscalus mexicanus*) 
---
**Capstone Project · DataTalks.Club 2026**  
**Author:** Victoria Torreno   
**Source:** [GBIF API](https://www.gbif.org/)  
**Goal:** Verify API connectivity, inspect raw JSON, and define the Bronze layer schema.

#### Setup

In [ ]:
import requests
import pandas as pd
import json

In [ ]:
url = "https://api.gbif.org/v1/occurrence/search"

In [ ]:
params = { 
    "scientificName": "Quiscalus mexicanus",
    "limit": 300 
}

#### API Request
Fetching 300 records from the GBIF occurrence search endpoint to inspect the raw response structure.

In [ ]:
response = requests.get(url, params=params)
data = response.json()

record_count = len(data['results'])
total_records = data['count']
print(f"Fetched {record_count} records out of {total_records:,} total sightings for `{params['scientificName']}`.")

#### Raw JSON Inspection  
Inspecting a single raw record to understand the full field structure before loading into a DataFrame.

In [ ]:
# inspect JSON structure of a record
print(json.dumps(data['results'][0], indent=2))

#### Schema & Field Overview
Loading records into a DataFrame to review available fields and data types.

In [ ]:
grackle_df = pd.DataFrame(data['results'])
# display top 5 rows
grackle_df.head()

In [ ]:
# check data types
grackle_df.dtypes

In [ ]:
# list all available columns
grackle_df.columns

#### Data Quality
Checking for nulls, missing coordinates, and basic statistics to identify fields requiring cleaning or dropping.

In [ ]:
# summary stats
grackle_df.describe()

In [ ]:
# check for any missing coordinates
grackle_df[['decimalLatitude', 'decimalLongitude']].isnull().mean()

In [ ]:
# null count per col
grackle_df.isnull().sum()

#### Geographic Coverage
Reviewing country distribution to confirm the dataset covers the species' expected range.

In [ ]:
# country distribution 
country_count = grackle_df['countryCode'].value_counts()
country_pct = (country_count / len(grackle_df) * 100).round(0).astype(int)
country_df = pd.DataFrame({'count': country_count, '%': country_pct}).rename_axis('country')

In [ ]:
country_df

#### Temporal Coverage
Checking date range and missing timestamps to define the pipeline's historical fetch window.

In [ ]:
# convert to datetime
grackle_df['eventDate'] = pd.to_datetime(grackle_df['eventDate'], errors='coerce')
# event date range
print(grackle_df['eventDate'].min(), 'to', grackle_df['eventDate'].max())

In [ ]:
# audit for null event dates
print(f"Missing timestamps: {grackle_df['eventDate'].isnull().sum()}")

#### Observation Types
Confirming record type distribution to determine whether filtering on `basisOfRecord` is needed in the pipeline.

In [ ]:
# observation type breakdown
grackle_df['basisOfRecord'].value_counts()

#### Data Exploration Summary 
**Target Species:** Great-tailed Grackle (*Quiscalus mexicanus*)  

---
**Volume**
- **Finding:** Total sightings exceed 3.7 million records.
- **Note:** The GBIF API has a 100k offset limit. Full ingestion will chunk requests by year to stay within this limit.

**Coordinate Coverage**
- **Finding:** 100% coordinate coverage in this 300-record sample.
- **Note:** Sample is clean, but a `dropna` step will be applied in the pipeline to handle missing coordinates in older records at scale.

**Data Quality**
- **Finding:** 95 fields total, many are sparse or near-empty (e.g. `vitality`, `occurrenceRemarks`, `informationWithheld`, `identificationRemarks`).
- **Note:** High null columns will be dropped during Bronze → Silver transformation to reduce schema noise.

**Geographic Coverage**
- **Finding:** Sightings span 11 countries, concentrated in the USA (52%) and Mexico (29%), consistent with the species' known range.
- **Note:** Central and South American records are present but sparse — expected given the species' primary range.

**Observation Types**
- **Finding:** 100% of records in this sample are `HUMAN_OBSERVATION`.
- **Note:** No filtering needed on `basisOfRecord` for this sample, but will validate this holds at scale in the full pipeline.